# Задача: Прогнозиране на заплата

## Въведение:
Вие сте младши Data Scientist в компания и получавате задача да подобрите съществуващ модел за прогнозиране на заплати. Настоящият модел е проста линейна регресия, която използва само няколко основни признака от данните, събрани от Glassdoor.
Вашата цел е да използвате уменията си в областта на **feature engineering** и **моделирането**, за да създадете значително по-точен модел.

## 1. Проблемът

Ще ви бъде предоставен Python скрипт, който:

1. Зарежда и извършва минимално почистване на данните.
2. Обучава базов модел (**Линейна регресия**) върху малък набор от характеристики.
3. Изчислява и отпечатва неговата средна абсолютна грешка (Mean Absolute Error - MAE).

Вашата задача е да работите в обозначената "Зона за състезатели", за да:

- Създадете нови, по-информативни характеристики от съществуващите данни.
- Изберете и обучите по-мощен модел.
- Постигнете по-нисък MAE от базовия модел.

## 2. Оценка

Основният показател за оценка е Средна абсолютна грешка (MAE). По-ниска стойност означава по-добър модел - вашата цел е да я минимизирате. MAE е лесна за интерпретиране, тъй като показва средната грешка в прогнозата в същите единици като заплатата (напр. MAE от 15 означава, че моделът греши средно с $15,000).

## 3. Предаване

Предайте генерираните си предсказания в submission_task_4_FirstName_LastName.csv върху данните от test_features.csv, заедно с тетрадката.



Зареждане на необходимите библиотеки

In [1]:
!pip install gdown
!gdown 1atCcqIxgKfvmtRUQkqz--wF2FTlEAHal
!gdown 1fA6iChCXW2e6v9a3G68r6b6L36_Em1ki

Downloading...
From: https://drive.google.com/uc?id=1atCcqIxgKfvmtRUQkqz--wF2FTlEAHal
To: /content/train.csv
100% 2.44M/2.44M [00:00<00:00, 138MB/s]
Downloading...
From: https://drive.google.com/uc?id=1fA6iChCXW2e6v9a3G68r6b6L36_Em1ki
To: /content/test_features.csv
100% 615k/615k [00:00<00:00, 115MB/s]


In [71]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
from sklearn.metrics import mean_absolute_error
import warnings
from sklearn.model_selection import GridSearchCV, cross_val_score

warnings.filterwarnings('ignore')

# Помощни функции и базов модел

In [72]:
# ==============================================================================
# === (ПОМОЩНИ ФУНКЦИИ И БАЗОВ МОДЕЛ - НЕ ПРОМЕНЯЙТЕ ТАЗИ КЛЕТКА) ===
# ==============================================================================

def title_simplifier(title):
    if 'data scientist' in title.lower():
        return 'data scientist'
    elif 'data engineer' in title.lower():
        return 'data engineer'
    elif 'analyst' in title.lower():
        return 'analyst'
    elif 'machine learning' in title.lower():
        return 'mle'
    elif 'manager' in title.lower():
        return 'manager'
    elif 'director' in title.lower():
        return 'director'
    else:
        return 'na'

def seniority(title):
    if 'sr' in title.lower() or 'senior' in title.lower() or 'lead' in title.lower() or 'principal' in title.lower():
        return 'senior'
    elif 'jr' in title.lower() or 'jr.' in title.lower():
        return 'jr'
    else:
        return 'na'

def load_data(path: str) -> pd.DataFrame:
    """
    Зарежда данните и извършва стабилно първоначално почистване,
    вдъхновено от подробния анализ.
    """
    df = pd.read_csv(path)
    return df

def clean_train_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df['Salary Estimate']!= '-1']

    # Почистване на стринга със заплатата
    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.split('(')[0])
    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.replace('K','').replace('$',''))

    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.lower().replace('per hour', ''))
    df['Salary Estimate'] = df['Salary Estimate'].apply(lambda x: x.lower().replace('employer provided salary:', ''))

    df['Rating'] = df['Rating'].apply(lambda x: x if x > 0 else np.nan)
    df['State'] = df.Location.apply(lambda x: x.split(',')[1])

    # Справяне както с диапазони (напр. "50-100"), така и с единични стойности ("75")
    df['Min_Salary'] = df['Salary Estimate'].apply(lambda x: int(x.split('-')[0]))
    df['Max_Salary'] = df['Salary Estimate'].apply(lambda x: int(x.split('-')[1]))
    df['Salary Estimate']= (df['Min_Salary'] + df['Max_Salary'])/2
    df.drop(['Min_Salary', 'Max_Salary'], axis=1, inplace=True)
    return df
def clean_test_data(df: pd.DataFrame) -> pd.DataFrame:

    df['Rating'] = df['Rating'].apply(lambda x: x if x > 0 else np.nan)
    df['State'] = df.Location.apply(lambda x: x.split(',')[1])

    return df

def clean_data_features(df: pd.DataFrame) -> pd.DataFrame:
    df['Rating'] = df['Rating'].apply(lambda x: x if x > 0 else np.nan)
    df['State'] = df.Location.apply(lambda x: x.split(',')[1])
    return df

# Зона за състезатели

## Цел:

Вашата задача е да имплементирате функцията student_solution по-долу. Целта е да създадете модел, който има по-ниска средна абсолютна грешка (MAE) от базовия модел.

## Стъпки:

1. Feature Engineering: Създайте нови, по-полезни характеристики.
2. Подготовка на данните: Изберете характеристиките, които ще използвате, и ги подгответе за модела по подходящ начин.
3. Обучение на модел: Изберете и обучете своя модел.

In [73]:
# ==============================================================================
# === (ГЛАВЕН СКРИПТ - НЕ ПРОМЕНЯЙТЕ ТАЗИ КЛЕТКА) ===
# ==============================================================================
# Зареждане на данните
df_train = load_data("train.csv")
df = clean_train_data(df_train)

In [75]:
!pip install catboost

In [76]:
# ==============================================================================
# ЗОНА ЗА ВАШИЯ КОД
# =============================================================================
# Може да опитате да:
# - почистите/преработите допълнително данните (напр. )
# - добавите други характеристики (напр. възраст на компанията)
# - експериментирате с различни модели
# - комбинирате няколко модела
# - пуснете grid search за намиране на оптимални параметри
# ==============================================================================

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

df_enhanced = df.copy()

df_enhanced['job_simp'] = df_enhanced['Job Title'].apply(title_simplifier)
df_enhanced['seniority'] = df_enhanced['Job Title'].apply(seniority)
df_enhanced['company_age'] = 2024 - df_enhanced['Founded']
df_enhanced['same_state'] = df_enhanced['Location'] == df_enhanced['Headquarters']  # В HQ заплатите са по-високи от колкото в други офиси :)

features = [
    'Rating', 'Size', 'Type of ownership', 'Industry', 'Sector', 'Revenue',
    'company_age', 'job_simp', 'seniority', 'same_state'
]
Y_column = ["Salary Estimate"]

df_model_train = df_enhanced[features + Y_column].dropna()

df_model_encoded = pd.get_dummies(df_model_train)

X_train_baseline = df_model_encoded.drop('Salary Estimate', axis=1)
y_train_baseline = df_model_encoded['Salary Estimate'].values

In [77]:
df_test = load_data("test_features.csv")
df_test = clean_test_data(df_test)

df_enhanced = df_test.copy()

df_enhanced['job_simp'] = df_enhanced['Job Title'].apply(title_simplifier)
df_enhanced['seniority'] = df_enhanced['Job Title'].apply(seniority)
df_enhanced['company_age'] = 2024 - df_enhanced['Founded']
df_enhanced['same_state'] = df_enhanced['Location'] == df_enhanced['Headquarters']  # В HQ заплатите са по-високи от колкото в други офиси :)

features = [
    'Rating', 'Size', 'Type of ownership', 'Industry', 'Sector', 'Revenue',
    'company_age', 'job_simp', 'seniority', 'same_state'
]

df_model_test = df_enhanced[features].dropna()

df_model_encoded = pd.get_dummies(df_model_test)

X_test = df_model_encoded

In [78]:
df_combined = pd.concat([df_model_train.drop('Salary Estimate', axis=1), df_model_test], axis=0, ignore_index=True)
df_combined_encoded = pd.get_dummies(df_combined)

# Възстановяване на train и test сега с еднакви колони
X_train_baseline = df_combined_encoded.iloc[:len(df_model_train), :]
y_train_baseline = df_model_train['Salary Estimate'].values

X_test = df_combined_encoded.iloc[len(df_model_train):, :]

In [79]:
X_train_baseline.columns

Index(['Rating', 'company_age', 'same_state', 'Size_1 to 50 employees',
       'Size_10000+ employees', 'Size_1001 to 5000 employees',
       'Size_201 to 500 employees', 'Size_5001 to 10000 employees',
       'Size_501 to 1000 employees', 'Size_51 to 200 employees',
       ...
       'job_simp_analyst', 'job_simp_data engineer', 'job_simp_data scientist',
       'job_simp_director', 'job_simp_manager', 'job_simp_mle', 'job_simp_na',
       'seniority_jr', 'seniority_na', 'seniority_senior'],
      dtype='object', length=129)

In [80]:
X_test.columns

Index(['Rating', 'company_age', 'same_state', 'Size_1 to 50 employees',
       'Size_10000+ employees', 'Size_1001 to 5000 employees',
       'Size_201 to 500 employees', 'Size_5001 to 10000 employees',
       'Size_501 to 1000 employees', 'Size_51 to 200 employees',
       ...
       'job_simp_analyst', 'job_simp_data engineer', 'job_simp_data scientist',
       'job_simp_director', 'job_simp_manager', 'job_simp_mle', 'job_simp_na',
       'seniority_jr', 'seniority_na', 'seniority_senior'],
      dtype='object', length=129)

In [81]:
param_grid = {
    'depth': [7, 8],
    'learning_rate': [0.125, 0.13],
    'iterations': [1700, 2500],
    'l2_leaf_reg': [0.425, 0.45]
}

cat_model = CatBoostRegressor(verbose=0, random_state=42)

grid_search = GridSearchCV(cat_model, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_search.fit(X_train_baseline, y_train_baseline)

GridSearchCV(cv=5,
             estimator=<catboost.core.CatBoostRegressor object at 0x7914c97a8150>,
             n_jobs=-1,
             param_grid={'depth': [7, 8], 'iterations': [1700, 2500],
                         'l2_leaf_reg': [0.425, 0.45],
                         'learning_rate': [0.125, 0.13]},
             scoring='neg_mean_absolute_error')

In [83]:
print(grid_search.best_estimator_.get_params())

{'loss_function': 'RMSE', 'verbose': 0, 'random_state': 42, 'depth': 8, 'iterations': 2500, 'l2_leaf_reg': 0.45, 'learning_rate': 0.125}


In [84]:

student_model = grid_search.best_estimator_

In [89]:
_, X_val, _, y_val = train_test_split(X_train_baseline, y_train_baseline, test_size=0.2, random_state=42)
student_predictions = student_model.predict(X_test)
y_pred_df = pd.DataFrame(student_predictions, columns=["Salary Estimate"])
#student_mae = mean_absolute_error(y_val, student_predictions)

print("\n--- Финални резултати ---")
print(f"Базовият модел е с MAE: {28.415}")
#print(f"Твоят модел е с MAE:    {student_mae:.3f}")

#improvement = 28.415 - student_mae
#if improvement > 0:
 #   print(f"\nПодобрение спрямо baseline модела: {improvement:.3f}!")
#else:
   # print("\nОпитай отново! Моделът ти не е по-добър от базовия.")

# Разкоментирайте, за да съхраните своите предсказания
first_name = "Mario"
last_name = "Petkov"
y_pred_df.to_csv(f'submission_task_4_{first_name}_{last_name}.csv', index=False)


--- Финални резултати ---
Базовият модел е с MAE: 28.415


MY MSE: 1.481